# The Text Only Model

This model cannot be feed images or vision data, just text.

![text only model](../showcase_images/text-only-model.png)

In [1]:
import easyjupyter
import torch.nn as nn
import torch
from model.decoder import Decoder
from model.generator import Generator
from model.RoPE import precompute_freqs_cis
from typing import Optional

In [2]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from configs import ConfigTemplate, Scaled_down_text

In [ ]:
class TextOnlyModel(nn.Module):
    def __init__(self, cfg: ConfigTemplate):
        """
        The text-only model. This model cannot be feed images or audio, only text.
        """
        super().__init__()
        self.cfg = cfg
        self.decoder = Decoder(cfg).to(cfg.device)

        self.token_embedding = nn.Embedding(
            num_embeddings=cfg.vocab_size, embedding_dim=cfg.d_model
        )

        self.generator = Generator(cfg)

        # Precompute RoPE frequencies
        self.freqs_cis = None
        self.compute_RoPE()
        self.initialize_weights()

    def compute_RoPE(self):
        """
        Compute RoPE representations. Once at the initialization, then again whenever the size of the seq_len changes, e.g., after the initial stage of pre-training the seq_len is increased.
        """
        self.freqs_cis = precompute_freqs_cis(
            cfg=self.cfg,
            head_dim=self.cfg.d_model // self.cfg.attn_heads,
            theta=self.cfg.rope_theta,
        ).to(self.cfg.device)

    def initialize_weights(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        print(
            f"\n\nModel initialized with {sum(p.numel() for p in self.parameters()):,} parameters!\n\n"
        )

    def forward(
        self,
        x,
        combined_mask: torch.Tensor,
        start_pos: int = 0,
        kv_cache: Optional[list[tuple]] = None,
    ):
        """
        Feed the model.
        Args:
            x: The input tensor of shape (batch_size, seq_len).
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            start_pos: Handles the slicing of the documents for both training and inference. During training, start_pos is always 0, because we process the full context window at once. During inference, start_pos increments by 1 for each new token the model generates.
            kv_caches: Inference only, a list of kv caches.
        
        Returns:
            logits, updated_kv_cache
        """
        # Make sure the sequence len doesn't exceed model's precomputed RoPE
        seq_len = x.shape[1]
        max_allowed_len = self.freqs_cis.shape[0]
        if start_pos + seq_len > max_allowed_len:
            raise ValueError(f"Sequence length ({start_pos + seq_len}) exceeds model's precomputed RoPE context window ({max_allowed_len}).")

        # Embed the input tokens
        x_embeds = self.token_embedding(x)

        # Slice the precomputed RoPE frequencies to match the seq_len, just in case the dataloader yields a smaller batch.
        batch_freqs_cis = self.freqs_cis[start_pos:start_pos+seq_len]

        # Run the Decoder
        decoder_out, updated_kv_cache = self.decoder(
            x_embeds, batch_freqs_cis, combined_mask, kv_cache
        )

        # Run the generator
        logits = self.generator(decoder_out)
        return logits, updated_kv_cache

`Test:`

In [ ]:
# @i-c
# TEST
from configs import Scaled_down_text
from model.utils.masking import create_causal_doc_mask

d_model = 512
vocab_size = 32000
seq_len = 256
batch_size = 8
cfg = Scaled_down_text.initialize()

dummy_input = torch.randint(0, vocab_size, (batch_size, seq_len), device=cfg.device)
mask = create_causal_doc_mask(cfg, dummy_input)

model = TextOnlyModel(cfg).to(cfg.device)
model(dummy_input, mask)


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: scaled_down_text


Model initialized with 1,034,027,520 parameters!




(tensor([[[ 5.8566e-02,  3.7896e-01, -2.6656e-01,  ...,  3.4601e-02,
           -3.5701e-01,  1.3414e-01],
          [ 8.0536e-02,  2.5766e-01, -3.2868e-01,  ..., -9.9478e-02,
           -2.6857e-01,  1.0766e-01],
          [ 3.2481e-02,  2.7556e-01, -3.3786e-01,  ..., -3.2844e-02,
           -1.5331e-01, -1.2457e-02],
          ...,
          [-1.8354e-01,  2.6150e-01,  1.6071e-01,  ...,  1.6230e-01,
           -2.5951e-01,  2.1155e-03],
          [-1.5829e-01,  2.0855e-01,  2.5253e-01,  ...,  2.0784e-01,
           -2.1104e-01,  4.7305e-02],
          [-9.4278e-02,  2.7384e-01,  1.3859e-01,  ...,  1.5165e-01,
           -2.8065e-01, -2.9755e-02]],
 
         [[ 3.6117e-01,  2.8785e-01, -8.4206e-02,  ..., -9.0924e-02,
           -7.8941e-02, -2.4510e-01],
          [ 3.3736e-01,  3.6069e-01, -8.3250e-02,  ..., -1.4349e-01,
            2.1500e-02, -3.0740e-01],
          [ 2.9614e-01,  3.6422e-01, -9.9356e-02,  ..., -1.2533e-01,
            5.2363e-02, -2.0405e-01],
          ...,
    

In [ ]:
# TODO use PreTrainedModel just need a class to transfer the pydantic config into one that HuggingFace can use.

In [3]:
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

class TextOnlyModel(PreTrainedModel):
    def __init__(self, cfg: ConfigTemplate):
        """
        The text-only model. This model cannot be feed images or audio, only text.
        """
        super().__init__()
        self.cfg = cfg
        self.decoder = Decoder(cfg).to(cfg.device)

        self.token_embedding = nn.Embedding(
            num_embeddings=cfg.vocab_size, embedding_dim=cfg.d_model
        )

        self.generator = Generator(cfg)

        # Precompute RoPE frequencies
        self.freqs_cis = None
        self.compute_RoPE()
        self.initialize_weights()

    def compute_RoPE(self):
        """
        Compute RoPE representations. Once at the initialization, then again whenever the size of the seq_len changes, e.g., after the initial stage of pre-training the seq_len is increased.
        """
        self.freqs_cis = precompute_freqs_cis(
            cfg=self.cfg,
            head_dim=self.cfg.d_model // self.cfg.attn_heads,
            theta=self.cfg.rope_theta,
        ).to(self.cfg.device)

    def initialize_weights(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        print(
            f"\n\nModel initialized with {sum(p.numel() for p in self.parameters()):,} parameters!\n\n"
        )

    def forward(
        self,
        x,
        combined_mask: torch.Tensor,
        start_pos: int = 0,
        kv_cache: Optional[list[tuple]] = None,
    ):
        """
        Feed the model.
        Args:
            x: The input tensor of shape (batch_size, seq_len).
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            start_pos: Handles the slicing of the documents for both training and inference. During training, start_pos is always 0, because we process the full context window at once. During inference, start_pos increments by 1 for each new token the model generates.
            kv_caches: Inference only, a list of kv caches.
        
        Returns:
            logits, updated_kv_cache
        """
        # Make sure the sequence len doesn't exceed model's precomputed RoPE
        seq_len = x.shape[1]
        max_allowed_len = self.freqs_cis.shape[0]
        if start_pos + seq_len > max_allowed_len:
            raise ValueError(f"Sequence length ({start_pos + seq_len}) exceeds model's precomputed RoPE context window ({max_allowed_len}).")

        # Embed the input tokens
        x_embeds = self.token_embedding(x)

        # Slice the precomputed RoPE frequencies to match the seq_len, just in case the dataloader yields a smaller batch.
        batch_freqs_cis = self.freqs_cis[start_pos:start_pos+seq_len]

        # Run the Decoder
        decoder_out, updated_kv_cache = self.decoder(
            x_embeds, batch_freqs_cis, combined_mask, kv_cache
        )

        # Run the generator
        logits = self.generator(decoder_out)
        return logits, updated_kv_cache

/Users/tonyavis/miniconda3/envs/build_an_llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# @i-c
# TEST
from configs import Scaled_down_text
from model.utils.masking import create_causal_doc_mask

d_model = 512
vocab_size = 32000
seq_len = 256
batch_size = 8
cfg = Scaled_down_text.initialize()

dummy_input = torch.randint(0, vocab_size, (batch_size, seq_len), device=cfg.device)
mask = create_causal_doc_mask(cfg, dummy_input)

model = TextOnlyModel(cfg).to(cfg.device)
model(dummy_input, mask)


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: scaled_down_text


TypeError: PreTrainedModel.__init__() missing 1 required positional argument: 'config'